# Food Price Watch 07: the monitor (Gradio app)
Builds `processed/monitor_8.csv` (per item and report day: price, the upper bound of each rule in Tk, and the alert flags of notebook 05b), checks it against 05b, then opens the monitor app.
The app is a replay of TCB prices to 24 Oct 2026: abnormal-rise detection, not early warning. Main rule: ML range; simple range and fixed rise can be switched on.
**`SHARE = True` makes a temporary public link (about 72 hours): anyone with it sees TCB prices** (approved by the user, DEVIATIONS.md). Set `AUTH = ('name', 'password')` to protect it, or `SHARE = False` to open it inside Colab only.
Paste the output of the check cell (the one before the app) back.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Code comes fresh from GitHub main each session; Drive holds data and outputs only.
import os
if not os.path.isdir('/content/foodprice_watch/.git'):
    !git clone --single-branch --branch main https://github.com/mrchowdhury587/foodprice_watch.git /content/foodprice_watch
else:
    !git -C /content/foodprice_watch pull --ff-only origin main
%pip install -q -r /content/foodprice_watch/requirements-colab.txt

In [ ]:
import sys, os, json
os.chdir('/content/foodprice_watch')
sys.path.insert(0, '/content/foodprice_watch/src')
CODE_COMMIT = !git rev-parse --short HEAD
CODE_COMMIT = CODE_COMMIT[0]
from foodwatch.paths import SETTINGS, DATA_ROOT, RAW, data, raw, sha256, write_csv
# stop if Drive is not really mounted (a failed mount leaves an empty /content/drive on the Colab machine)
assert os.path.ismount('/content/drive'), 'Google Drive is not mounted: Runtime > Disconnect and delete runtime, then rerun cell 1'
assert DATA_ROOT.is_dir(), f'{DATA_ROOT} not found in Google Drive (wrong account, or the folder is missing)'
for folder in ['raw', 'processed', 'reports/tables']:
    (DATA_ROOT / folder).mkdir(parents=True, exist_ok=True)
for tmp in DATA_ROOT.rglob('*.tmp'):   # half-written files from an interrupted session
    tmp.unlink()
print('code commit:', CODE_COMMIT, '| data root:', DATA_ROOT)

In [ ]:
%pip install -q "gradio>=5.40,<6"
import gradio
print('gradio', gradio.__version__)

In [ ]:
import numpy as np
import pandas as pd
from foodwatch import monitor as mo
from foodwatch.paths import REPO
choice = SETTINGS['alert_choice']
D = pd.read_csv(data('data'), parse_dates=['Date'])
F7 = pd.read_csv(data('forecasts_h7'))
M, EP = mo.build(F7, D, SETTINGS, choice)
AL = pd.read_csv(data('alerts'))
R5 = pd.read_csv(REPO / 'reports/tables/report/R5_alert_rules.csv')
chk = mo.checks(M, AL, R5, SETTINGS, choice)
assert chk['ok'], chk
h_m = write_csv(M, data('monitor'))

In [ ]:
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print('=== 07 monitor data ===')
print('code commit:', CODE_COMMIT, '| forecasts_h7.csv sha256', sha256(data('forecasts_h7'))[:16], '| alerts_8.csv sha256', sha256(data('alerts'))[:16])
print('checks:', 'PASS' if chk['ok'] else 'FAIL', {k_: v_ for k_, v_ in chk.items() if k_ != 'ok'})
print('monitor file:', SETTINGS['files']['monitor'], '| rows', len(M), '| sha256', h_m[:16])
last = M.date.max().strftime('%Y-%m-%d')
for r_ in 'ABC':
    T = mo.status_on(M, last, r_)
    print(f"status on {last}, {mo.RULES[r_]}: {int((T.Status == 'ALERT: abnormal rise').sum())} of {len(T)} items in alert")
print(mo.status_on(M, last, SETTINGS['monitor']['main_rule']).drop(columns=['Price (Tk)', 'Upper bound (Tk)']).to_string(index=False))

In [ ]:
# Opens the app. SHARE = True prints a public *.gradio.live link (about 72 hours); anyone with the link sees TCB prices.
SHARE = True
AUTH = None          # e.g. ('supervisor', 'a password you choose') to require a login
sys.path.insert(0, str(REPO / 'app'))
from monitor_app import make_app, load
app = make_app(*load(DATA_ROOT, REPO))
app.launch(share=SHARE, auth=AUTH)